# Multi-Agent Trading Analysis

This notebook demonstrates the `LangGraph`-based multi-agent trading desk.

The pipeline consists of:
1. **Market Context** — fetches current market regime context
2. **Macro Regime Detector** — classifies regime (risk_on / neutral / risk_off / crisis)
3. **Parallel Analysts** — fundamental (CFA), sentiment (alt-data), technical (CMT), earnings
4. **Debate Loop** — configurable rounds of agent debate
5. **Risk Manager** — veto power over final decision
6. **Portfolio Strategist** — synthesises into `AnalysisResult`

An `ANTHROPIC_API_KEY` is required for real LLM calls. Without it, nodes use
deterministic stubs so the graph still executes end-to-end.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
repo_root = Path('.').resolve().parent
sys.path.insert(0, str(repo_root))
print(f'Project root: {repo_root}')


In [ ]:
# Import the main analysis function
from src.agents.graph import analyze_ticker, build_trading_desk_graph
from src.utils.schemas import AnalysisResult, Decision

print('analyze_ticker function loaded successfully.')
print(f'Decision options: {[d.value for d in Decision]}')


In [ ]:
# Run analysis for AAPL (requires ANTHROPIC_API_KEY for real LLM; stubs run without it)
ticker = 'AAPL'
print(f'Analyzing {ticker}...')

try:
    result = analyze_ticker(ticker, qlib_context='Alpha score: 2.3% 20d momentum')
    print(f'\nAnalysis complete!')
    print(f'  Ticker    : {result.ticker}')
    print(f'  Decision  : {result.decision.value}')
    print(f'  Confidence: {result.confidence:.1f}%')
    print(f'  Reasoning : {result.reasoning[:200]}...' if len(result.reasoning) > 200 else f'  Reasoning : {result.reasoning}')
except Exception as e:
    print(f'Analysis error: {e}')
    # Create a stub result for demonstration
    result = AnalysisResult(
        ticker=ticker,
        decision=Decision.HOLD,
        confidence=50.0,
        reasoning='Stub result — LLM unavailable (no API key)',
    )
    print(f'  Using stub result: {result.decision.value}')


In [ ]:
# Display the full AnalysisResult
print('Full AnalysisResult:')
print(f'  Ticker          : {result.ticker}')
print(f'  Asset Class     : {result.asset_class.value}')
print(f'  Decision        : {result.decision.value}')
print(f'  Confidence      : {result.confidence:.1f}%')
print(f'  Target Price    : {result.target_price}')
print(f'  Stop Loss       : {result.stop_loss}')
print(f'  Position Size   : {result.position_size_pct:.1f}%')
print(f'  Time Horizon    : {result.time_horizon}')
print(f'  Risk Flags      : {result.risk_flags}')
print(f'  Catalysts       : {result.catalysts}')
print(f'  Agent Reports   : {list(result.agent_reports.keys())}')
print(f'  Timestamp       : {result.timestamp}')


## Summary

`analyze_ticker(ticker, qlib_context)` runs the full LangGraph pipeline and returns an `AnalysisResult`.

Key parameters:
- `ticker` — stock or crypto ticker (e.g. `"AAPL"`, `"BTC/USDT"`)
- `asset_class` — `"EQUITY"` or `"CRYPTO"`
- `qlib_context` — optional Qlib alpha scores string to inject into the analysis

The `AnalysisResult` schema includes:
- `decision`: STRONG_BUY / BUY / HOLD / SELL / STRONG_SELL
- `confidence`: 0-100%
- `target_price`, `stop_loss`: optional price levels
- `position_size_pct`: recommended portfolio weight
- `agent_reports`: per-agent reasoning breakdown
- `risk_flags`, `catalysts`: structured risk and opportunity lists

Set `ANTHROPIC_API_KEY` in `config/.env` for real LLM calls.
